# 5. Origin of the degenerate predicted yields (Table S7)

Four pairs of solvents among the 29 candidates receive **identical** predicted
yields. Because 1,2-dichlorobenzene and chlorobenzene were both examined
experimentally and gave markedly different yields (99% and 72%), it matters
whether the degeneracy comes from the descriptor or from the training set.

This notebook shows that

1. the MACCS Keys fingerprint **does** distinguish every pair (they differ in one to
   three bits, and the Tanimoto coefficients are well below 1), but
2. **every one of those discriminating bits is invariant across the ten training
   solvents**, so variance-based standardization reduces them to zero and the
   regression assigns them zero weight.

The degeneracy is therefore a property of the structural coverage of the training
set, not of the fingerprint.

**Output:** `output/table_s7.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MACCSkeys import GenMACCSKeys
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

N_COMPONENTS = 3

def maccs_matrix(smiles_list):
    """Convert a list of SMILES into a (n_molecules, 167) MACCS Keys matrix."""
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    if any(m is None for m in mols):
        bad = [s for s, m in zip(smiles_list, mols) if m is None]
        raise ValueError(f"RDKit could not parse: {bad}")
    return np.array([GenMACCSKeys(m) for m in mols], dtype=float)

from rdkit.Chem import rdFingerprintGenerator, DataStructs
from rdkit.Chem.MACCSkeys import smartsPatts

## Reproduce the predictions for all 29 candidate solvents

Same model as in `0_fingerprint_and_pls.ipynb`.

In [ ]:
solvent = pd.read_csv(DATA / "solvent.csv")
train = solvent.dropna(subset=["exp_yield"])

x_scaler, y_scaler = StandardScaler(), StandardScaler()
X_train = x_scaler.fit_transform(maccs_matrix(train["smiles"]))
y_train = y_scaler.fit_transform(train[["exp_yield"]].to_numpy(float))
model = PLSRegression(N_COMPONENTS).fit(X_train, y_train)

X_all = maccs_matrix(solvent["smiles"])
solvent["pred_yield"] = y_scaler.inverse_transform(
    model.predict(x_scaler.transform(X_all))).ravel()

# bits that vary across the ten training solvents; all other bits are
# standardized to zero and therefore carry no regression weight
X_train_raw = maccs_matrix(train["smiles"])
varying_bits = set(np.where(X_train_raw.min(axis=0) != X_train_raw.max(axis=0))[0].tolist())
print(f"{len(varying_bits)} of 167 bits vary across the training solvents")

## Find the degenerate pairs automatically

Predictions are grouped by their value rounded to four decimals; any group with
more than one member is a degenerate set.

In [ ]:
groups = solvent.groupby(solvent["pred_yield"].round(4))["name"].apply(list)
degenerate = [names for names in groups if len(names) > 1]

for names in degenerate:
    value = solvent.loc[solvent["name"] == names[0], "pred_yield"].iloc[0]
    print(f"{value:8.4f} %  <-  {' / '.join(names)}")

## Analyse each pair

For every pair we list

* the MACCS bits in which the two molecules differ, with the SMARTS pattern that
  defines each bit (`rdkit.Chem.MACCSkeys.smartsPatts`),
* how many of those bits actually vary across the ten training solvents,
* the Tanimoto coefficient computed with MACCS Keys and with ECFP4 (2048 bits),
  which shows that both descriptors distinguish the pair when the full fingerprint
  is considered.

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)  # ECFP4

rows = []
for names in degenerate:
    for a, b in zip(names[:-1], names[1:]):
        rec_a = solvent[solvent["name"] == a].iloc[0]
        rec_b = solvent[solvent["name"] == b].iloc[0]
        mol_a, mol_b = Chem.MolFromSmiles(rec_a["smiles"]), Chem.MolFromSmiles(rec_b["smiles"])
        fp_a, fp_b = GenMACCSKeys(mol_a), GenMACCSKeys(mol_b)

        diff = np.where(np.array(fp_a) != np.array(fp_b))[0]
        described = ", ".join(f"{i} ({smartsPatts[i][0]})" for i in diff)
        n_varying = sum(1 for i in diff if i in varying_bits)

        rows.append({
            "Solvent pair": f"{a} / {b}",
            "Predicted yield (%)": round(float(rec_a["pred_yield"]), 1),
            "MACCS bits that differ (SMARTS)": described,
            "Bits varying in the training set": f"{n_varying} of {len(diff)}",
            "Tanimoto (MACCS)": round(DataStructs.TanimotoSimilarity(fp_a, fp_b), 3),
            "Tanimoto (ECFP4)": round(DataStructs.TanimotoSimilarity(
                morgan.GetFingerprint(mol_a), morgan.GetFingerprint(mol_b)), 3),
        })

table_s7 = pd.DataFrame(rows)
table_s7.to_csv(OUT / "table_s7.csv", index=False)
table_s7

## Interpretation

Every discriminating bit falls in the "0 of *n*" category: none of them varies
across the ten training solvents. The model is blind to precisely those structural
features that would separate the members of each pair.

The degeneracy is therefore **self-diagnosing and can be used constructively**: it
identifies the structural features on which the current training data are
uninformative, and thereby indicates which additional experiment would be most
informative. Including a single solvent that carries the relevant bit restores the
ability of the model to discriminate between the members of the pair.